In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(REPO_ROOT / 'src'))

from hotelling.spatial.census import build_grid_polygons
from hotelling.spatial.assembly import (
    build_demand_grid,
    enrich_supermarkets_with_brw,
)

PATH_RAW       = REPO_ROOT / 'data' / 'raw'
PATH_PROCESSED = REPO_ROOT / 'data' / 'processed'

# Load grid (convert point midpoints to square polygons)
grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')
grid = build_grid_polygons(grid)
grid['index'] = grid.index
print(f"Grid: {len(grid)} cells")

# Load all spatial layer inputs
grid_malls         = gpd.read_parquet(PATH_PROCESSED / 'grid_malls.parquet')
grid_with_stations = gpd.read_parquet(PATH_PROCESSED / 'grid_with_stations.parquet')
travel_times       = pd.read_parquet(PATH_PROCESSED / 'travel_times.parquet')
employment_clusters= gpd.read_parquet(PATH_PROCESSED / 'employment_clusters.parquet')
supermarkets       = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet')
brw                = gpd.read_file(PATH_RAW / 'brw_2025.gpkg')
print("All inputs loaded.")

In [ ]:
# Assemble demand grid: flags + employment + travel times + MSS/ESIx + normalization
demand_grid = build_demand_grid(
    grid=grid,
    grid_malls=grid_malls,
    grid_with_stations=grid_with_stations,
    travel_times=travel_times,
    employment_clusters=employment_clusters,
    mss_path=PATH_RAW / 'mss.gpkg',
    esix_path=PATH_RAW / 'esix.gpkg',
    output_path=PATH_PROCESSED / 'demand_grid.parquet',
)

# Enrich incumbents with BRW data
supermarkets_full = enrich_supermarkets_with_brw(
    supermarkets=supermarkets,
    brw=brw,
    output_path=PATH_PROCESSED / 'supermarkets_full.parquet',
)

print(f"demand_grid: {len(demand_grid)} cells, {len(demand_grid.columns)} columns")
print(f"supermarkets_full: {len(supermarkets_full)} stores")

In [ ]:
demand_grid